# Compartment evaluation

This notebook evaluates compartment profiles from raw, scHiCluster-imputed and Higashi-imputed Hi-C data. The retained example uses the original Tan/Neuron settings and saved outputs.


## 1. Data preparation

This section defines the dataset, target cell type, resolution and file paths, then reads the cached compartment calls. The three dictionaries hold single-cell profiles, pseudo-bulk profiles and raw pseudo-bulk PCA reference tracks.


### 1.1. Imports and configuration


In [1]:
import re
import pickle
import cooler
import warnings
from tqdm import tqdm
from scipy.stats import pearsonr
from tools import *
warnings.filterwarnings("ignore")

In [2]:
dataset = 'Tan' # dataset
target_cell_type = 'Neuron' # cell type
is_denoise = True
label_method = 'pca_raw'
resolution = 100000

data_types = ['raw', 'schicluster', 'higashi']
methods = ['pca', 'cpg', 'higashi']

# ------------ bulk_num  ------------------
sc_label_file = f"/home/zzl/data/Higashi/temp/{dataset}/100K_cool/label_info.pickle"
with open(sc_label_file, "rb") as f:
    label_info = pickle.load(f)
sc_cell_type_info = label_info["cell type"]
sc_cell_index_list = [i for i, ct in enumerate(sc_cell_type_info) if ct == target_cell_type]

total = len(sc_cell_index_list)
print('total',total)
merge_nums = list(range(100, total + 1, 100))
if merge_nums[-1] != total:
    merge_nums.append('all')
print('merge_nums',merge_nums)

def get_bulk_num_paths(num, type):
    if type == 'higashi':
        base = f'/home/zzl/data/Higashi/temp/{dataset}/100K_bulk_num/exp1_nbr_5_impute/{num}'
        hdf5_base = f'scCompartment_bulk{num}.hdf5'
    elif type == 'schicluster':
        base = f'/home/zzl/data/schicluster/data/{dataset}/100K_bulk_num/{num}'
        hdf5_base = f'scCompartment_bulk{num}_scHiCluster.hdf5'
    elif type == 'raw':
        base = f'/home/zzl/data/Higashi/temp/{dataset}/100K_bulk_num/raw/{num}'
        hdf5_base = f'scCompartment_bulk{num}_raw.hdf5'

    pca = f'{dataset}.{type}.pca.bulk{num}.compartment.nc'
    cpg = f'{dataset}.{type}.cpg.bulk{num}.compartment.nc'
    higashi = os.path.join(f'/home/zzl/data/Higashi/temp/{dataset}/100K_bulk_num/',hdf5_base)

    return base, higashi, pca, cpg

def get_bulk_cell_index(bulk_dir, target_cell_type, merge_num, type):
    target_cell_type = re.sub(r'[\\/:*?"<>|]', '_', target_cell_type)
    table = pd.read_csv(os.path.join(bulk_dir, 'cell_table_100K.tsv'), sep='\t', header=None)
    names = table.iloc[:, 0].tolist()
    if type !='raw':
        return names.index(f'{target_cell_type}_bulk_num_{merge_num}')
    else:
        return names.index(f'{target_cell_type}')

if dataset in ['Lee','Tian']:
    chrom_size_path = '/home/zzl/schicluster/files/chromsize/hg38.autosomal.chrom.sizes'
    cpg_file = '/home/zzl/Higashi/higashi/cpg_hg38.txt'
    chromosomes_to_use = ["chr1","chr2","chr3","chr4","chr5",
			  "chr6","chr7","chr8","chr9","chr10",
			  "chr11","chr12","chr13","chr14","chr15",
			  "chr16","chr17","chr18","chr19","chr20","chr21","chr22"]
else:
    chrom_size_path = '/home/zzl/schicluster/files/chromsize/mm10.chrom.sizes'
    cpg_file = '/home/zzl/Higashi/higashi/cpg_mm10.txt'
    chromosomes_to_use = ["chr1","chr2","chr3","chr4","chr5",
			  "chr6","chr7","chr8","chr9","chr10",
			  "chr11","chr12","chr13","chr14","chr15",
			  "chr16","chr17","chr18","chr19"]

chrom_sizes_all = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes_all.loc[chromosomes_to_use]

total 1260
merge_nums [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 'all']


In [ ]:
filename = f"/home/zzl/data/schicluster/all_data/{dataset}_{target_cell_type.replace('/', '_')}.pkl"
data_to_save = {
    'single_cell_call': {},
    'bulk_num_call': {},
    'raw_bulk_num_pca_call': {}
}
with open(filename, 'wb') as f:
    pickle.dump(data_to_save, f)
print(f"Data saved to {filename}")

### 1.2. Read data

In [5]:
with open(filename, 'rb') as f:
    loaded_data = pickle.load(f)

single_cell_call = loaded_data['single_cell_call']
bulk_num_call = loaded_data['bulk_num_call']
raw_bulk_num_pca_call = loaded_data['raw_bulk_num_pca_call']

## 2. Boundary and track evaluation

Single-cell and pseudo-bulk profiles are compared with the same cell type's all-cell raw PCA reference. The notebook summarizes boundary F1, boundary counts and Pearson track correlations for each caller and input modality.


### 2.1. Single cell

In [71]:
def compute_single_cell_overall_metrics():
    if label_method == 'pca_raw':
        golden_label_source = raw_bulk_num_pca_call
    else:
        raise NotImplementedError(f"label_method '{label_method}' not supported yet.")

    result_dict = {method: {} for method in methods}

    for method in methods:
        for datatype in data_types:
            all_cell_f1s = []
            all_cell_boundary_counts = []
            all_cell_corrs = []

            for cell_idx in tqdm(range(len(sc_cell_index_list)), desc=f"{method}|{datatype}|Processing cells"):
                chr_f1s = []
                chr_boundary_counts = []
                chr_corrs = []

                for chr_name in chromosomes_to_use:
                    try:
                        pred_vec = single_cell_call[method][datatype][chr_name][cell_idx]  # (n_bins,)
                    except (KeyError, IndexError):
                        print(f"    Warning: Missing data for {method}/{datatype}/{chr_name}/cell{cell_idx}")
                        continue

                    # --- 1. F1 Score ---
                    ref_key = f"{chr_name}_bulkall"
                    if ref_key in golden_label_source:
                        label_vec = golden_label_source[ref_key]
                        try:
                            _, _, f1, _ = boundary_overlap_metrics(pred_vec, label_vec, denoise=is_denoise)
                            if np.isfinite(f1) and f1 >= 0:
                                chr_f1s.append(f1)
                        except Exception as e:
                            print(f"    Error computing F1 for {chr_name}, cell {cell_idx}: {e}")
                            continue

                    # --- 2. Boundary Count ---
                    try:
                        denoised_vec = denoise_ab_vector(pred_vec)
                        n_bound = len(find_boundaries(denoised_vec))
                        chr_boundary_counts.append(n_bound)
                    except Exception as e:
                        pass

                    # # --- 3. label Correlation ---
                    try:
                        r, _ = pearsonr(denoised_vec, label_vec)
                        chr_corrs.append(r)
                    except Exception as e:
                        pass
                if chr_f1s:
                    all_cell_f1s.append(np.mean(chr_f1s))
                if chr_boundary_counts:
                    all_cell_boundary_counts.append(np.mean(chr_boundary_counts))
                if chr_corrs:
                    all_cell_corrs.append(np.mean(chr_corrs))

            result_dict[method][datatype] = {
                'f1': np.mean(all_cell_f1s),
                'boundary_count': np.mean(all_cell_boundary_counts),
                'corr': np.nanmean(all_cell_corrs)
            }
            print(result_dict[method][datatype])

    return result_dict

In [73]:
results = compute_single_cell_overall_metrics()

pca|raw|Processing cells: 100%|██████████| 1198/1198 [00:44<00:00, 26.63it/s]


{'f1': 0.053112873916011084, 'boundary_count': 29.579474562867937, 'corr': 0.03842473947349981}


pca|schicluster|Processing cells: 100%|██████████| 1198/1198 [00:58<00:00, 20.53it/s]


{'f1': 0.1324542372570242, 'boundary_count': 191.20775854494332, 'corr': 0.107772091895227}


pca|higashi|Processing cells: 100%|██████████| 1198/1198 [00:52<00:00, 22.78it/s]


...

{'f1': 0.30174435387454573, 'boundary_count': 40.914155170898866, 'corr': 0.7464033417015056}


higashi|schicluster|Processing cells: 100%|██████████| 1198/1198 [00:47<00:00, 25.33it/s]


{'f1': 0.21302348306173657, 'boundary_count': 70.93928477286707, 'corr': 0.4875225677523689}


higashi|higashi|Processing cells: 100%|██████████| 1198/1198 [00:57<00:00, 20.82it/s]

{'f1': 0.2978615938458857, 'boundary_count': 53.66545119058079, 'corr': 0.7106366727542782}


In [74]:
df = pd.DataFrame({
    (method, dt): results[method][dt]
    for method in methods for dt in data_types
}).T
df

f1  boundary_count      corr
pca     raw          0.053113       29.579475  0.038425
        schicluster  0.132454      191.207759  0.107772
        higashi      0.237979       57.797733  0.660212
cpg     raw          0.186675       83.262279  0.602333
        schicluster  0.171641       98.759380  0.586542
        higashi      0.255243       44.780116  0.624192
higashi raw          0.301744       40.914155  0.746403
        schicluster  0.213023       70.939285  0.487523
        higashi      0.297862       53.665451  0.710637

### 2.2. Bulk level

In [6]:
def compute_bulk_overall_metrics(bulknum=None):
    bulk_str = str(bulknum)
    if label_method == 'pca_raw':
        golden_label_source = raw_bulk_num_pca_call
    else:
        raise NotImplementedError(f"label_method '{label_method}' not supported yet.")

    result_dict = {method: {} for method in methods}

    for method in methods:
        for datatype in data_types:
            chr_f1s = []
            chr_boundary_counts = []
            chr_corrs = []

            for chr_name in tqdm(chromosomes_to_use, desc=f"{method}|{datatype}|chr"):
                key = f"{chr_name}_bulk{bulk_str}"
                try:
                    pred_vec = bulk_num_call[method][datatype][key]  # (n_bins,)
                except KeyError:
                    print(f"    Warning: Missing data for {method}/{datatype}/{key}")
                    continue

                # --- 1. F1 Score vs golden label (bulkall) ---
                ref_key = f"{chr_name}_bulkall"
                if ref_key in golden_label_source:
                    label_vec = golden_label_source[ref_key]
                    try:
                        _, _, f1, _ = boundary_overlap_metrics(pred_vec, label_vec, denoise=is_denoise)
                        if np.isfinite(f1) and f1 >= 0:
                            chr_f1s.append(f1)
                    except Exception as e:
                        print(f"    Error computing F1 for {chr_name}: {e}")
                        pass

                # --- 2. Boundary Count ---
                try:
                    denoised_vec = denoise_ab_vector(pred_vec)
                    n_bound = len(find_boundaries(denoised_vec))
                    chr_boundary_counts.append(n_bound)
                except Exception as e:
                    pass

                # --- 3. CpG Correlation ---
                try:
                    r, _ = pearsonr(denoised_vec, label_vec)
                    chr_corrs.append(r)
                except Exception as e:
                    pass

            result_dict[method][datatype] = {
                'f1': np.nanmean(chr_f1s),
                'boundary_count': np.nanmean(chr_boundary_counts),
                'corr': np.nanmean(chr_corrs)
            }

            print(f"{method}|{datatype} => F1: {result_dict[method][datatype]['f1']:.4f}, "
                  f"Boundary: {result_dict[method][datatype]['boundary_count']:.2f}, "
                  f"Corr: {result_dict[method][datatype]['corr']:.4f}")

    return result_dict

In [8]:
bulk_num = 100

In [9]:
bulkall_metrics = compute_bulk_overall_metrics(bulknum=bulk_num)

pca|raw|chr: 100%|██████████| 19/19 [00:00<00:00, 218.50it/s]


pca|raw => F1: 0.5040, Boundary: 56.89, Corr: 0.7814


pca|schicluster|chr: 100%|██████████| 19/19 [00:00<00:00, 595.12it/s]


pca|schicluster => F1: 0.4885, Boundary: 63.95, Corr: 0.8626


pca|higashi|chr: 100%|██████████| 19/19 [00:00<00:00, 659.34it/s]


...

higashi|raw => F1: 0.2770, Boundary: 38.79, Corr: 0.8092


higashi|schicluster|chr: 100%|██████████| 19/19 [00:00<00:00, 528.21it/s]


higashi|schicluster => F1: 0.2940, Boundary: 44.05, Corr: 0.8617


higashi|higashi|chr: 100%|██████████| 19/19 [00:00<00:00, 482.76it/s]

higashi|higashi => F1: 0.3016, Boundary: 45.37, Corr: 0.8583


In [10]:
df = pd.DataFrame({
    (method, dt): bulkall_metrics[method][dt]
    for method in methods for dt in data_types
}).T
df

f1  boundary_count      corr
pca     raw          0.504006       56.894737  0.781426
        schicluster  0.488542       63.947368  0.862573
        higashi      0.306384       42.368421  0.828175
cpg     raw          0.277417       38.789474  0.680657
        schicluster  0.245323       29.263158  0.672543
        higashi      0.226676       27.947368  0.811660
higashi raw          0.277004       38.789474  0.809227
        schicluster  0.294009       44.052632  0.861720
        higashi      0.301636       45.368421  0.858290

## 3. Saddle strength

The corresponding raw or imputed contact matrix provides the observed/expected map. Compartment scores define the saddle groups; within- and between-compartment enrichment are then compared for single-cell and pseudo-bulk samples.


### 3.1. Single cell

In [11]:
def load_matrix_from_cool_or_raw(chr_name, file_path, imputation_type, cell_index=None):
    if imputation_type == 'raw':
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"{file_path} not found.")
        Q = np.load(file_path, allow_pickle=True)[cell_index]
        Q = Q.toarray()
    else:
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"{file_path} not found.")
        c = cooler.Cooler(file_path)
        Q = c.matrix(balance=False).fetch(chr_name)

    return Q

In [13]:
def compute_single_cell_strength(metric='saddle strength'):
    if metric == 'saddle strength':
        metric_key = 'saddle_strength'
    elif metric in ['AA', 'BB', 'AB', 'BA']:
        metric_key = metric
    else:
        raise ValueError("metric must be one of: 'saddle strength', 'AA', 'BB', 'AB', 'BA'")

    def get_sc_matrix_path(datatype):
        if datatype == 'raw':
            return f'/home/zzl/data/Higashi/temp/{dataset}/100K_cool/raw/'
        elif datatype == 'higashi':
            return f'/home/zzl/data/Higashi/temp/{dataset}/100K_cool/exp1_nbr_5_impute/'
        elif datatype == 'schicluster':
            return f'/home/zzl/data/schicluster/data/{dataset}/100K/'
        else:
            raise ValueError(f"Unsupported datatype: {datatype}")

    result_dict = {method: {} for method in methods}

    for method in methods:
        for datatype in data_types:
            all_cell_metrics = []

            for cell_idx in tqdm(range(len(sc_cell_index_list)), desc=f"{method}|{datatype}|Cells"):
                chr_metrics = []

                for chr_name in chromosomes_to_use:
                    try:
                        # --- 1. compartment---
                        ab_vec = single_cell_call[method][datatype][chr_name][cell_idx]

                        # --- 2. contact matrix ---
                        sc_base = get_sc_matrix_path(datatype)

                        if datatype == 'raw':
                            file_path = os.path.join(sc_base, f"{chr_name}_sparse_adj.npy")
                            contact_mat = load_matrix_from_cool_or_raw(chr_name, file_path, imputation_type=datatype, cell_index=cell_idx)
                        else:
                            cell_table_path = os.path.join(sc_base, 'cell_table_100K.tsv')
                            cell_table = pd.read_csv(cell_table_path, sep='\t', header=None)
                            cool_files = [f for f in cell_table[1].tolist() if os.path.exists(f) and f.endswith('.cool')]
                            if cell_idx >= len(cool_files):
                                print(f"Warning: cell_idx {cell_idx} out of range for {datatype} cool files")
                                continue
                            file_path = cool_files[cell_idx]
                            contact_mat = load_matrix_from_cool_or_raw(chr_name, file_path, imputation_type=datatype, cell_index=None)

                        # --- 3.OE ---
                        oe_mat = get_OE_matrix(contact_mat)

                        # --- 4.saddle ---
                        saddle_result = compute_saddle(oe_mat, ab_vec)

                        # --- 5. metric ---
                        value = saddle_result[metric_key]
                        if np.isfinite(value):
                            chr_metrics.append(value)
                        else:
                            print(f"Warning: NaN {metric} for {chr_name}, cell {cell_idx}")

                    except Exception as e:
                        print(f"Error processing {chr_name}, cell {cell_idx}: {e}")
                        continue

                if chr_metrics:
                    all_cell_metrics.append(np.mean(chr_metrics))

            result_dict[method][datatype] = np.nanmean(all_cell_metrics)
            print(f"{method}|{datatype} => {metric}: {result_dict[method][datatype]:.4f}")

    return result_dict

In [ ]:
ss = compute_single_cell_strength(metric='saddle strength')

### 3.2. Bulk level

In [15]:
def compute_bulk_strength(bulknum=None, metric='saddle strength'):
    if metric == 'saddle strength':
        metric_key = 'saddle_strength'
    elif metric in ['AA', 'BB', 'AB', 'BA']:
        metric_key = metric
    else:
        raise ValueError("metric must be one of: 'saddle strength', 'AA', 'BB', 'AB', 'BA'")

    bulk_str = str(bulknum)
    def get_bulk_matrix_path(datatype, merge_num):
        if datatype == 'raw':
            return f'/home/zzl/data/Higashi/temp/{dataset}/100K_bulk_num/raw/{merge_num}/'
        elif datatype == 'higashi':
            return f'/home/zzl/data/Higashi/temp/{dataset}/100K_bulk_num/exp1_nbr_5_impute/{merge_num}/'
        elif datatype == 'schicluster':
            return f'/home/zzl/data/schicluster/data/{dataset}/100K_bulk_num/{merge_num}/'
        else:
            raise ValueError(f"Unsupported datatype: {datatype}")

    bulk_cell_index_cache = {}
    for datatype in data_types:
        bulk_cell_index_cache[datatype] = {}
        for mn in [bulknum]:
            bulk_base, _, _, _ = get_bulk_num_paths(mn, type=datatype)
            idx = get_bulk_cell_index(bulk_base, target_cell_type, mn, type=datatype)
            bulk_cell_index_cache[datatype][mn] = idx

    result_dict = {method: {} for method in methods}

    for method in methods:
        for datatype in data_types:
            chr_metrics = []

            for chr_name in tqdm(chromosomes_to_use, desc=f"{method}|{datatype}|Chr"):
                key = f"{chr_name}_bulk{bulk_str}"

                try:
                    ab_vec = bulk_num_call[method][datatype][key]

                    bulk_dir = get_bulk_matrix_path(datatype, bulknum)
                    # chr_bins = chr_bins_dict[chr_name]
                    cell_idx = bulk_cell_index_cache[datatype][bulknum]

                    if datatype == 'raw':
                        file_path = os.path.join(bulk_dir, f"{chr_name}_sparse_adj.npy")
                        contact_mat = load_matrix_from_cool_or_raw(
                            chr_name, file_path, imputation_type=datatype, cell_index=cell_idx
                        )
                    else:
                        cell_table_path = os.path.join(bulk_dir, 'cell_table_100K.tsv')
                        if not os.path.exists(cell_table_path):
                            raise FileNotFoundError(f"Cell table missing: {cell_table_path}")
                        cell_table = pd.read_csv(cell_table_path, sep='\t', header=None)
                        cool_files = [f for f in cell_table[1].tolist() if os.path.exists(f) and f.endswith('.cool')]
                        if cell_idx >= len(cool_files):
                            print(f"    Warning: cell_idx {cell_idx} out of range for bulk {bulknum} {datatype}")
                            continue
                        file_path = cool_files[cell_idx]
                        contact_mat = load_matrix_from_cool_or_raw(
                            chr_name, file_path, imputation_type=datatype, cell_index=None
                        )

                    oe_mat = get_OE_matrix(contact_mat)

                    saddle_result = compute_saddle(oe_mat, ab_vec)

                    value = saddle_result[metric_key]
                    if np.isfinite(value):
                        chr_metrics.append(value)
                    else:
                        print(f"Warning: NaN {metric} for {chr_name}, bulk{bulk_str}")

                except KeyError:
                    print(f"Warning: Missing data for {method}/{datatype}/{key}")
                    continue
                except Exception as e:
                    print(f"Error processing {chr_name}, bulk{bulk_str}: {e}")
                    continue

            result_dict[method][datatype] = np.nanmean(chr_metrics)

    return result_dict

In [16]:
bulk_ss = compute_bulk_strength(bulknum=bulk_num, metric='saddle strength')

pca|raw|Chr: 100%|██████████| 19/19 [00:05<00:00,  3.76it/s]
pca|schicluster|Chr: 100%|██████████| 19/19 [00:04<00:00,  4.50it/s]
pca|higashi|Chr: 100%|██████████| 19/19 [00:04<00:00,  4.44it/s]
cpg|raw|Chr: 100%|██████████| 19/19 [00:02<00:00,  7.31it/s]
cpg|schicluster|Chr: 100%|██████████| 19/19 [00:04<00:00,  4.58it/s]
cpg|higashi|Chr: 100%|██████████| 19/19 [00:04<00:00,  4.29it/s]
higashi|raw|Chr: 100%|██████████| 19/19 [00:02<00:00,  7.19it/s]
higashi|schicluster|Chr: 100%|██████████| 19/19 [00:04<00:00,  4.36it/s]
higashi|higashi|Chr: 100%|██████████| 19/19 [00:04<00:00,  4.63it/s]


In [17]:
rows = []
for data_type, methods in bulk_ss.items():
    for method, saddle_strength in methods.items():
        rows.append([data_type, method, saddle_strength])

df = pd.DataFrame(rows, columns=['data_type', 'method', 'saddle_strength'])
df

,data_type,method,saddle_strength
0,pca,raw,14.194105
1,pca,schicluster,4.882393
2,pca,higashi,6.046843
3,cpg,raw,9.920650
4,cpg,schicluster,3.717176
5,cpg,higashi,4.591673
6,higashi,raw,16.592420
7,higashi,schicluster,5.252330
8,higashi,higashi,6.248656
